# Prompt-only Baseline

Use this notebook to run the no-post-training baseline on `phase1_test.jsonl`.

Recommended default:

- model: `Qwen/Qwen2.5-3B-Instruct`
- input: `data/samples/phase1_test.jsonl`
- output: `results/predictions/qwen25_3b_prompt_only_test.jsonl`

In [1]:
# Run this once in a fresh Jupyter environment if dependencies are missing.

%pip install -q transformers datasets accelerate pyyaml jsonschema

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import os


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'configs').exists() and (candidate / 'data').exists():
            return candidate
    raise FileNotFoundError('Could not locate project root from current working directory.')


PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)

PROJECT_ROOT = /home/lyan11/small-llm-structured-posttraining


In [3]:
import torch
import platform
import sys

print('python =', sys.version)
print('platform =', platform.platform())
print('cuda_available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu =', torch.cuda.get_device_name(0))
    print('gpu_count =', torch.cuda.device_count())
    print('bf16_supported =', torch.cuda.is_bf16_supported())

python = 3.13.11 | packaged by conda-forge | (main, Dec  6 2025, 11:24:03) [GCC 14.3.0]
platform = Linux-6.8.0-106-generic-x86_64-with-glibc2.39
cuda_available = True
gpu = NVIDIA H200 NVL
gpu_count = 1
bf16_supported = True


In [4]:
from pathlib import Path
import json

base_model = 'Qwen/Qwen2.5-3B-Instruct'
test_path = PROJECT_ROOT / 'data' / 'samples' / 'phase1_test.jsonl'
prediction_dir = PROJECT_ROOT / 'results' / 'predictions'
prediction_dir.mkdir(parents=True, exist_ok=True)
prediction_path = prediction_dir / 'qwen25_3b_prompt_only_test.jsonl'

print('test_exists =', test_path.exists(), test_path)
print('prediction_path =', prediction_path)

test_records = []
with test_path.open('r', encoding='utf-8') as handle:
    for line in handle:
        test_records.append(json.loads(line))

len(test_records), test_records[0]

test_exists = True /home/lyan11/small-llm-structured-posttraining/data/samples/phase1_test.jsonl
prediction_path = /home/lyan11/small-llm-structured-posttraining/results/predictions/qwen25_3b_prompt_only_test.jsonl


(254,
 {'sample_id': 'console_ai_it_helpdesk_tickets-000409-sai67ttf1',
  'task_name': 'ticket_structured_output',
  'schema_name': 'ticket_schema_v1',
  'complexity_bucket': 'simple',
  'input_text': "Subject: Password Reset Assistance Needed\nDescription: Hi IT, I need help resetting my password for our video conferencing software. I forgot it and can't log in. I tried using the 'Forgot Password' link, but it didn't work. My colleague was able to reset theirs without any issues.",
  'target_json': {'ticket_id': 'sai67ttf1',
   'summary': 'Password Reset Assistance Needed',
   'category': 'bug',
   'priority': 'medium',
   'requires_followup': True,
   'reporter': {'name': 'Jane Doe', 'team': None},
   'affected_systems': [{'name': 'Software', 'component': 'software'}],
   'actions_requested': [{'action': 'Investigate issue: Password Reset Assistance Needed',
     'owner': None,
     'deadline': None}],
   'constraints': {'environment': None, 'blocking': None}},
  'metadata': {'source

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
dtype = torch.bfloat16 if use_bf16 else torch.float16

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=dtype,
    device_map='auto',
    trust_remote_code=True,
)
model.eval()
print('model loaded')

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

model loaded


In [6]:
generation_kwargs = {
    'max_new_tokens': 256,
    'do_sample': False,
    'temperature': 1.0,
    'top_p': 1.0,
}

generation_kwargs

{'max_new_tokens': 256, 'do_sample': False, 'temperature': 1.0, 'top_p': 1.0}

In [7]:
def build_inference_messages(record):
    return [
        {
            'role': 'system',
            'content': 'You are an information extraction model. Return only JSON that matches the requested schema. Do not add explanations or markdown.',
        },
        {
            'role': 'user',
            'content': (
                f"Task: extract a structured record for {record['task_name']}.\n"
                f"Schema name: {record['schema_name']}\n"
                'Return a JSON object only.\n'
                'Input text:\n'
                f"{record['input_text']}"
            ),
        },
    ]


def generate_prediction_text(record):
    messages = build_inference_messages(record)
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt_text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, **generation_kwargs)
    generated_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()


sample_pred = generate_prediction_text(test_records[0])
print(sample_pred)

The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


{
  "subject": "Password Reset Assistance Needed",
  "description": "Hi IT, I need help resetting my password for our video conferencing software. I forgot it and can't log in. I tried using the 'Forgot Password' link, but it didn't work. My colleague was able to reset theirs without any issues."
}


In [8]:
predictions = []

for idx, record in enumerate(test_records):
    prediction_text = generate_prediction_text(record)
    prediction_json = None
    try:
        prediction_json = json.loads(prediction_text)
    except json.JSONDecodeError:
        prediction_json = None

    predictions.append(
        {
            'sample_id': record['sample_id'],
            'prediction_text': prediction_text,
            'prediction_json': prediction_json,
            'metadata': {
                'model_name': base_model,
                'experiment_id': 'qwen25_3b_prompt_only_v1',
            },
        }
    )

    if (idx + 1) % 25 == 0:
        print(f'generated {idx + 1} / {len(test_records)}')

len(predictions)

generated 25 / 254
generated 50 / 254
generated 75 / 254
generated 100 / 254
generated 125 / 254
generated 150 / 254
generated 175 / 254
generated 200 / 254
generated 225 / 254
generated 250 / 254


254

In [9]:
with prediction_path.open('w', encoding='utf-8') as handle:
    for record in predictions:
        handle.write(json.dumps(record, ensure_ascii=False) + '\n')

print('saved predictions to', prediction_path)
print('exists =', prediction_path.exists())
print('size =', prediction_path.stat().st_size if prediction_path.exists() else None)

saved predictions to /home/lyan11/small-llm-structured-posttraining/results/predictions/qwen25_3b_prompt_only_test.jsonl
exists = True
size = 254496


In [10]:
with prediction_path.open('r', encoding='utf-8') as handle:
    for i, line in enumerate(handle):
        if i >= 2:
            break
        print(json.loads(line))
        print('---')

{'sample_id': 'console_ai_it_helpdesk_tickets-000409-sai67ttf1', 'prediction_text': '{\n  "subject": "Password Reset Assistance Needed",\n  "description": "Hi IT, I need help resetting my password for our video conferencing software. I forgot it and can\'t log in. I tried using the \'Forgot Password\' link, but it didn\'t work. My colleague was able to reset theirs without any issues."\n}', 'prediction_json': {'subject': 'Password Reset Assistance Needed', 'description': "Hi IT, I need help resetting my password for our video conferencing software. I forgot it and can't log in. I tried using the 'Forgot Password' link, but it didn't work. My colleague was able to reset theirs without any issues."}, 'metadata': {'model_name': 'Qwen/Qwen2.5-3B-Instruct', 'experiment_id': 'qwen25_3b_prompt_only_v1'}}
---
{'sample_id': 'console_ai_it_helpdesk_tickets-000017-xka1oq83l', 'prediction_text': '{\n  "subject": "Updating Personal Profile on Intranet",\n  "description": "Hi Team, Can you please pr